[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/42_label_smoothing_solution.ipynb)

# 🟡 Solution: Label Smoothing Loss

Reference solution using the stable logsumexp form of log-softmax.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F


In [ ]:
# ✅ SOLUTION

def label_smoothing_loss(logits: torch.Tensor, targets: torch.Tensor,
                         smoothing: float = 0.1, reduction: str = "mean") -> torch.Tensor:
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    batch_idx = torch.arange(targets.numel(), device=targets.device)
    nll = -log_probs[batch_idx, targets]
    smooth_loss = -log_probs.mean(dim=-1)
    loss = (1.0 - smoothing) * nll + smoothing * smooth_loss

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unknown reduction: {reduction}")


In [ ]:
# Verify
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print("Smoothed loss:", label_smoothing_loss(logits, targets, smoothing=0.1))
print("CE ref:       ", F.cross_entropy(logits, targets))
print("Per-example:  ", label_smoothing_loss(logits, targets, smoothing=0.1, reduction="none"))


In [ ]:
# Run judge
from torch_judge import check
check('label_smoothing')
